# Olist E-commerce: DML, DDL, Joins and Aggregates in PostgreSQL

## 0. Setup

In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pgserver') is None:
  subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pgserver', 'sqlalchemy>=2.0,<2.1', 'psycopg2-binary', 'pandas'],
    check=True,
  )

In [3]:
import os
import urllib.request
import zipfile

import pandas as pd
from pgserver.postgres_server import get_server
from sqlalchemy import create_engine, text

pg = get_server('/tmp/pg_data')
engine = create_engine(pg.get_uri(), isolation_level='AUTOCOMMIT', connect_args={'options': '-c timezone=UTC'})
conn = engine.connect()

conn.exec_driver_sql('SELECT version()').scalar()

'PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit'

In [6]:
def run_sql(sql: str) -> None:
  conn.execute(text(sql))


def query(sql: str) -> pd.DataFrame:
  result = conn.execute(text(sql))

  return pd.DataFrame(result.fetchall(), columns=list(result.keys()))


def copy_csv(table: str, path: str, options: str = 'FORMAT csv, HEADER true') -> int:
  raw_conn = engine.raw_connection()
  cursor = raw_conn.cursor()
  with open(path, encoding='utf-8') as csv_file:
    cursor.copy_expert(f'COPY {table} FROM STDIN WITH ({options})', csv_file)
  loaded_rows = cursor.rowcount
  raw_conn.commit()
  raw_conn.close()

  return loaded_rows

## 1. Data source and raw layer

In [8]:
import csv

OLIST_DIR = os.path.join('data', 'olist')
OLIST_MIRROR = 'https://raw.githubusercontent.com/Softjey/goit-rdb-hw-04/main/data/olist'
OLIST_FILES = [
  'olist_customers_dataset.csv',
  'olist_orders_dataset.csv',
  'olist_order_items_dataset.csv',
  'olist_products_dataset.csv',
  'olist_sellers_dataset.csv',
  'olist_order_reviews_dataset.csv',
  'product_category_name_translation.csv',
]

os.makedirs(OLIST_DIR, exist_ok=True)
for file_name in OLIST_FILES:
  path = os.path.join(OLIST_DIR, file_name)
  if not os.path.exists(path):
    urllib.request.urlretrieve(f'{OLIST_MIRROR}/{file_name}', path)


def load_raw(table: str, path: str) -> int:
  with open(path, encoding='utf-8-sig') as csv_file:
    header = next(csv.reader(csv_file))
  columns = ', '.join(f'{column} TEXT' for column in header)
  run_sql(f'DROP TABLE IF EXISTS {table} CASCADE; CREATE TABLE {table} ({columns})')

  return copy_csv(table, path)


{name: os.path.getsize(os.path.join(OLIST_DIR, name)) for name in OLIST_FILES}

{'olist_customers_dataset.csv': 9033957,
 'olist_orders_dataset.csv': 17654914,
 'olist_order_items_dataset.csv': 15438671,
 'olist_products_dataset.csv': 2379446,
 'olist_sellers_dataset.csv': 174703,
 'olist_order_reviews_dataset.csv': 14451670,
 'product_category_name_translation.csv': 2613}

In [10]:
raw_files = {
  'olist_customers_raw': 'olist_customers_dataset.csv',
  'olist_orders_raw': 'olist_orders_dataset.csv',
  'olist_order_items_raw': 'olist_order_items_dataset.csv',
  'olist_products_raw': 'olist_products_dataset.csv',
  'olist_sellers_raw': 'olist_sellers_dataset.csv',
  'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
  'olist_category_translation_raw': 'product_category_name_translation.csv',
}

{table: load_raw(table, os.path.join(OLIST_DIR, file_name)) for table, file_name in raw_files.items()}

{'olist_customers_raw': 99441,
 'olist_orders_raw': 99441,
 'olist_order_items_raw': 112650,
 'olist_products_raw': 32951,
 'olist_sellers_raw': 3095,
 'olist_order_reviews_raw': 99224,
 'olist_category_translation_raw': 71}

## 2. Typed schema

In [11]:
run_sql('''
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS seller_score CASCADE;
DROP TABLE IF EXISTS seller_alerts CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
DROP TABLE IF EXISTS olist_order_reviews CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;
DROP TABLE IF EXISTS olist_category_translation CASCADE;

CREATE TABLE olist_customers (
  customer_id TEXT PRIMARY KEY,
  customer_unique_id TEXT NOT NULL,
  customer_zip_code_prefix INTEGER,
  customer_city TEXT,
  customer_state CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
  order_id TEXT PRIMARY KEY,
  customer_id TEXT NOT NULL REFERENCES olist_customers (customer_id),
  order_status TEXT NOT NULL,
  order_purchase_timestamp TIMESTAMP NOT NULL,
  order_approved_at TIMESTAMP,
  order_delivered_carrier_date TIMESTAMP,
  order_delivered_customer_date TIMESTAMP,
  order_estimated_delivery_date TIMESTAMP,
  CHECK (order_status IN (
    'created', 'approved', 'invoiced', 'processing',
    'shipped', 'delivered', 'unavailable', 'canceled'
  ))
);

CREATE TABLE olist_category_translation (
  product_category_name TEXT PRIMARY KEY,
  product_category_name_english TEXT NOT NULL
);

CREATE TABLE olist_products (
  product_id TEXT PRIMARY KEY,
  product_category_name TEXT,
  product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
  product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
  product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
  product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
  product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
  product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
  product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
  seller_id TEXT PRIMARY KEY,
  seller_zip_code_prefix INTEGER,
  seller_city TEXT,
  seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
  order_id TEXT NOT NULL REFERENCES olist_orders (order_id),
  order_item_id INTEGER NOT NULL,
  product_id TEXT REFERENCES olist_products (product_id),
  seller_id TEXT REFERENCES olist_sellers (seller_id),
  shipping_limit_date TIMESTAMP,
  price NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
  freight_value NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
  PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
  review_row_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  review_id TEXT,
  order_id TEXT NOT NULL REFERENCES olist_orders (order_id),
  review_score SMALLINT CHECK (review_score BETWEEN 1 AND 5),
  review_comment_title TEXT,
  review_comment_message TEXT,
  review_creation_date TIMESTAMP,
  review_answer_timestamp TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews (order_id);
''')

In [22]:
run_sql('''
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
  order_id,
  customer_id,
  order_status,
  order_purchase_timestamp::TIMESTAMP,
  order_approved_at::TIMESTAMP,
  order_delivered_carrier_date::TIMESTAMP,
  order_delivered_customer_date::TIMESTAMP,
  order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_category_translation
SELECT product_category_name, product_category_name_english
FROM olist_category_translation_raw;

INSERT INTO olist_products
SELECT
  product_id,
  product_category_name,
  product_name_lenght::INTEGER,
  product_description_lenght::INTEGER,
  product_photos_qty::INTEGER,
  product_weight_g::INTEGER,
  product_length_cm::INTEGER,
  product_height_cm::INTEGER,
  product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
  order_id,
  order_item_id::INTEGER,
  product_id,
  seller_id,
  shipping_limit_date::TIMESTAMP,
  price::NUMERIC(12, 2),
  freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
  review_id, order_id, review_score, review_comment_title,
  review_comment_message, review_creation_date, review_answer_timestamp
)
SELECT
  review_id,
  order_id,
  review_score::SMALLINT,
  review_comment_title,
  review_comment_message,
  review_creation_date::TIMESTAMP,
  review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
''')

query('''
SELECT 'customers' AS table_name, (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows, (SELECT COUNT(*) FROM olist_customers) AS typed_rows
UNION ALL SELECT 'orders', (SELECT COUNT(*) FROM olist_orders_raw), (SELECT COUNT(*) FROM olist_orders)
UNION ALL SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw), (SELECT COUNT(*) FROM olist_order_items)
UNION ALL SELECT 'products', (SELECT COUNT(*) FROM olist_products_raw), (SELECT COUNT(*) FROM olist_products)
UNION ALL SELECT 'sellers', (SELECT COUNT(*) FROM olist_sellers_raw), (SELECT COUNT(*) FROM olist_sellers)
UNION ALL SELECT 'reviews', (SELECT COUNT(*) FROM olist_order_reviews_raw), (SELECT COUNT(*) FROM olist_order_reviews)
UNION ALL SELECT 'category_translation', (SELECT COUNT(*) FROM olist_category_translation_raw), (SELECT COUNT(*) FROM olist_category_translation)
ORDER BY table_name
''')

,table_name,raw_rows,typed_rows
0,category_translation,71,71
1,customers,99441,99441
2,order_items,112650,112650
3,orders,99441,99441
4,products,32951,32951
5,reviews,99224,99224
6,sellers,3095,3095


All seven typed tables have exactly as many rows as the raw layer: 99,441 customers and orders, 112,650 items, 32,951 products, 3,095 sellers, 99,224 reviews and 71 category translations. No row was dropped, because all foreign keys, `CHECK`s and type casts passed on the full dataset. `COPY` loads empty CSV cells as `NULL` right away, so `::TIMESTAMP` and `::INTEGER` do not fail on gaps.

In [23]:
query('''
SELECT
  COUNT(*) AS review_rows,
  COUNT(DISTINCT review_id) AS distinct_review_ids,
  COUNT(DISTINCT order_id) AS reviewed_orders,
  COUNT(*) FILTER (WHERE review_comment_message IS NULL) AS rows_without_message,
  (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL) AS products_without_category,
  (SELECT COUNT(*) FROM olist_products AS p WHERE p.product_category_name IS NOT NULL
     AND NOT EXISTS (SELECT 1 FROM olist_category_translation AS t WHERE t.product_category_name = p.product_category_name)) AS products_without_translation
FROM olist_order_reviews
''')

,review_rows,distinct_review_ids,reviewed_orders,rows_without_message,products_without_category,products_without_translation
0,99224,98410,98673,58247,610,13


`review_id` cannot be a primary key: 99,224 rows have only 98,410 distinct `review_id` values, and 789 of them are linked to several orders. Another 547 orders have several reviews, so the technical `review_row_id` and aggregation to `order_id` are mandatory. The review text is empty in 58,247 rows (59%). In addition, 610 products have no category, and 13 products have a category without an English translation (see `EXCEPT` below).

## 3. DML with PostgreSQL features

### 3.1 INSERT ... RETURNING

In [24]:
run_sql('''
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
  alert_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  seller_id TEXT NOT NULL REFERENCES olist_sellers (seller_id),
  alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
  severity SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
  details TEXT,
  created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
)
''')

query('''
WITH seller_orders AS (
  SELECT DISTINCT oi.seller_id, oi.order_id
  FROM olist_order_items AS oi
),
late_sellers AS (
  SELECT so.seller_id, COUNT(DISTINCT o.order_id) AS late_orders
  FROM seller_orders AS so
  JOIN olist_orders AS o ON o.order_id = so.order_id
  WHERE o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
  GROUP BY so.seller_id
  ORDER BY late_orders DESC, so.seller_id
  LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT seller_id, 'late_delivery', 4, 'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at
''')

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-11-08 14:06:09.870262+00:00


The query finds the seller with the most late orders (172) and creates an alert for them. `seller_orders` first reduces items to `seller × order` pairs, so an order with several items from one seller is not counted several times. `RETURNING` immediately returns the generated `alert_id`, `created_at` and the details text. That row can go into an audit log or be passed down the pipeline without a separate `SELECT` after the insert.

### 3.2 INSERT ... ON CONFLICT DO UPDATE

In [45]:
run_sql('''
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
  seller_id TEXT PRIMARY KEY REFERENCES olist_sellers (seller_id),
  avg_rating NUMERIC(3, 2),
  n_reviews INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
  tier TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
  updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
)
''')

SELLER_SCORE_UPSERT = '''
WITH seller_orders AS (
  SELECT DISTINCT oi.seller_id, oi.order_id
  FROM olist_order_items AS oi
),
review_per_order AS (
  SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
  FROM olist_order_reviews AS rv
  WHERE rv.review_score IS NOT NULL
  GROUP BY rv.order_id
),
seller_rating AS (
  SELECT
    so.seller_id,
    AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
    COUNT(rpo.order_id) AS n_reviews
  FROM seller_orders AS so
  LEFT JOIN review_per_order AS rpo ON rpo.order_id = so.order_id
  GROUP BY so.seller_id
),
upserted AS (
  INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
  SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
      WHEN avg_rating IS NULL THEN NULL
      WHEN avg_rating >= 4.5 THEN 'gold'
      WHEN avg_rating >= 3.5 THEN 'silver'
      ELSE 'bronze'
    END
  FROM seller_rating
  ON CONFLICT (seller_id) DO UPDATE
  SET avg_rating = EXCLUDED.avg_rating,
    n_reviews = EXCLUDED.n_reviews,
    tier = EXCLUDED.tier,
    updated_at = NOW()
  RETURNING seller_id, avg_rating, n_reviews, tier, updated_at
)
SELECT COALESCE(tier, 'no reviews') AS tier, COUNT(*) AS sellers, ROUND(AVG(avg_rating), 2) AS avg_rating, SUM(n_reviews) AS reviewed_orders
FROM upserted
GROUP BY tier
ORDER BY sellers DESC
'''

query(SELLER_SCORE_UPSERT)

,tier,sellers,avg_rating,reviewed_orders
0,silver,1568,4.06,87123
1,gold,983,4.83,7093
2,bronze,539,2.30,5031
3,no reviews,5,None,0


In [47]:
query(SELLER_SCORE_UPSERT)

,tier,sellers,avg_rating,reviewed_orders
0,silver,1568,4.06,87123
1,gold,983,4.83,7093
2,bronze,539,2.30,5031
3,no reviews,5,None,0


In [48]:
query('''
WITH naive AS (
  SELECT oi.seller_id, COUNT(rv.review_score) AS naive_reviews, AVG(rv.review_score)::NUMERIC(3, 2) AS naive_avg
  FROM olist_order_items AS oi
  LEFT JOIN olist_order_reviews AS rv ON rv.order_id = oi.order_id
  GROUP BY oi.seller_id
)
SELECT
  (SELECT COUNT(*) FROM seller_score) AS seller_score_rows,
  (SELECT COUNT(DISTINCT seller_id) FROM olist_order_items) AS sellers_with_items,
  SUM(s.n_reviews) AS correct_reviews,
  SUM(n.naive_reviews) AS naive_reviews,
  COUNT(*) FILTER (WHERE s.avg_rating IS DISTINCT FROM n.naive_avg) AS sellers_with_different_avg
FROM seller_score AS s
JOIN naive AS n USING (seller_id)
''')

,seller_score_rows,sellers_with_items,correct_reviews,naive_reviews,sellers_with_different_avg
0,3095,3095,99247,112372,1264


`seller_orders` uses `SELECT DISTINCT` because one row of `olist_order_items` is one item, and an order with three items from a seller would produce three rows. Reviews are first aggregated to `order_id`, because 547 orders have several of them, and only then joined to the `seller × order` pair. The check shows that the naive join `order_items → reviews` gives 112,372 "reviews" instead of 99,247 and changes the average rating for 1,264 of 3,095 sellers.

`ON CONFLICT (seller_id) DO UPDATE` makes the run repeatable: the second cell with the same query added no rows, because every seller already has a row and it is only updated. The table has 3,095 rows, one per seller. 5 sellers have only orders without reviews: `AVG` over only `NULL`s gives `NULL`, `COUNT(rpo.order_id)` gives 0, and `CASE` returns `NULL` instead of a tier so as not to invent a rating.

Limitation: a review in Olist belongs to an order, not to a seller. 1,278 orders contain items from several sellers, and their score is credited to each of them.

### 3.3 UPDATE ... FROM

In [52]:
run_sql('''
ALTER TABLE olist_orders ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE
''')

query('''
WITH delivery_flags AS (
  SELECT
    order_id,
    (
      order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
      AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
    ) AS is_late_calc
  FROM olist_orders
),
updated AS (
  UPDATE olist_orders AS o
  SET is_late = f.is_late_calc
  FROM delivery_flags AS f
  WHERE f.order_id = o.order_id
  RETURNING o.order_id, o.order_status, o.is_late
)
SELECT is_late, COUNT(*) AS orders, COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered_orders
FROM updated
GROUP BY is_late
ORDER BY is_late
''')

,is_late,orders,delivered_orders
0,False,92906,89944
1,True,6535,6534


`UPDATE ... FROM` takes the flag from the CTE `delivery_flags` and rewrites `is_late` for every order, and `RETURNING` inside the CTE lets the result be counted right away. There are 6,535 late orders, 6,534 of them delivered. The query is idempotent because the value is computed from the dates every time instead of accumulating, so a repeat run gives the same distribution. The column is added with `ADD COLUMN IF NOT EXISTS`, so re-running the cell does not fail either.

### 3.4 DELETE ... RETURNING

In [56]:
query('''
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at
''')

,alert_id,seller_id,alert_type,created_at


In [57]:
run_sql('''
INSERT INTO seller_alerts (seller_id, alert_type, severity, details, created_at)
SELECT seller_id, 'data_quality', 2, 'Backfilled test alert', NOW() - INTERVAL '45 days'
FROM seller_alerts
WHERE alert_type = 'late_delivery'
''')

query('''
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at
''')

,alert_id,seller_id,alert_type,created_at
0,2,4a3ca9315b744ce9f8e9374361493884,data_quality,2026-09-24 14:06:11.289773+00:00


The first `DELETE` returned an empty result, which is expected: the only alert was created seconds ago, and the rule deletes records older than 30 days. To show that the rule works, a test alert dated 45 days ago is inserted next, and then `DELETE ... RETURNING` returns exactly that one (`alert_id = 2`). `RETURNING` acts as a log here: it shows what was deleted, and that set can be saved before the data disappears.

## 4. Customer segments

In [58]:
run_sql('''
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
  customer_unique_id TEXT PRIMARY KEY,
  latest_customer_id TEXT REFERENCES olist_customers (customer_id),
  customer_state CHAR(2),
  customer_city TEXT
);

INSERT INTO olist_customer_dim (customer_unique_id, latest_customer_id, customer_state, customer_city)
SELECT DISTINCT ON (c.customer_unique_id)
  c.customer_unique_id,
  c.customer_id,
  c.customer_state,
  c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o ON o.customer_id = c.customer_id
ORDER BY c.customer_unique_id, o.order_purchase_timestamp DESC NULLS LAST, c.customer_id
''')

query('''
SELECT
  (SELECT COUNT(*) FROM olist_customers) AS customer_ids,
  (SELECT COUNT(*) FROM olist_customer_dim) AS unique_customers,
  (SELECT COUNT(*) FROM (SELECT customer_unique_id FROM olist_customers GROUP BY customer_unique_id HAVING COUNT(*) > 1) AS repeat) AS customers_with_several_ids
''')

,customer_ids,unique_customers,customers_with_several_ids
0,99441,96096,2997


In [83]:
run_sql('''
CREATE TABLE customer_segments (
  segment_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  customer_unique_id TEXT NOT NULL,
  segment_name TEXT NOT NULL CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),
  rfm_score SMALLINT NOT NULL CHECK (rfm_score BETWEEN 1 AND 5),
  recency_days INTEGER CHECK (recency_days IS NULL OR recency_days >= 0),
  monetary_value NUMERIC(12, 2) NOT NULL DEFAULT 0 CHECK (monetary_value >= 0),
  n_orders INTEGER NOT NULL DEFAULT 0 CHECK (n_orders >= 0),
  assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
  ADD CONSTRAINT fk_customer_segments_customer_unique
  FOREIGN KEY (customer_unique_id)
  REFERENCES olist_customer_dim (customer_unique_id)
  ON DELETE CASCADE
''')

SEGMENTS_UPSERT = '''
WITH reference_date AS (
  SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
  FROM olist_orders
),
order_totals AS (
  SELECT
    o.order_id,
    o.customer_id,
    o.order_purchase_timestamp::DATE AS order_date,
    COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
  FROM olist_orders AS o
  LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
  GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp
),
customer_stats AS (
  SELECT
    c.customer_unique_id,
    COUNT(DISTINCT ot.order_id) AS n_orders,
    COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
    MAX(ot.order_date) AS last_order_date
  FROM olist_customer_dim AS cd
  JOIN olist_customers AS c ON c.customer_unique_id = cd.customer_unique_id
  LEFT JOIN order_totals AS ot ON ot.customer_id = c.customer_id
  GROUP BY c.customer_unique_id
),
segmented AS (
  SELECT
    cs.customer_unique_id,
    cs.n_orders,
    cs.monetary_value,
    CASE WHEN cs.last_order_date IS NULL THEN NULL ELSE (rd.as_of_date - cs.last_order_date)::INTEGER END AS recency_days,
    CASE
      WHEN cs.monetary_value >= 1000 THEN 'VIP'
      WHEN cs.last_order_date IS NOT NULL AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90 AND cs.n_orders = 1 THEN 'new'
      WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
      ELSE 'inactive'
    END AS segment_name,
    CASE
      WHEN cs.monetary_value >= 1000 THEN 5
      WHEN cs.monetary_value >= 500 THEN 4
      WHEN cs.n_orders >= 2 THEN 3
      WHEN cs.n_orders = 1 THEN 2
      ELSE 1
    END::SMALLINT AS rfm_score
  FROM customer_stats AS cs
  CROSS JOIN reference_date AS rd
),
upserted AS (
  INSERT INTO customer_segments (customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders)
  SELECT customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
  FROM segmented
  ON CONFLICT (customer_unique_id) DO UPDATE
  SET segment_name = EXCLUDED.segment_name,
    rfm_score = EXCLUDED.rfm_score,
    recency_days = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders = EXCLUDED.n_orders,
    assigned_at = NOW()
  RETURNING customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
)
SELECT
  segment_name,
  COUNT(*) AS customers,
  ROUND(AVG(n_orders), 2) AS avg_orders,
  ROUND(AVG(monetary_value), 2) AS avg_monetary,
  ROUND(AVG(recency_days)) AS avg_recency_days
FROM upserted
GROUP BY segment_name
ORDER BY customers DESC
'''

query(SEGMENTS_UPSERT)

,segment_name,customers,avg_orders,avg_monetary,avg_recency_days
0,regular,45982,1.07,224.57,307
1,inactive,40032,1.00,59.28,316
2,new,8891,1.00,140.39,72
3,VIP,1191,1.11,1591.81,290


In [85]:
query(SEGMENTS_UPSERT)

query('SELECT COUNT(*) AS segment_rows, COUNT(DISTINCT customer_unique_id) AS customers FROM customer_segments')

,segment_rows,customers
0,96096,96096


In [86]:
query('''
SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname
''')

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


`customer_id` in Olist is created per order: 99,441 `customer_id` values belong to only 96,096 real buyers, and 2,997 of them have several ids. If RFM were computed per `customer_id`, every "customer" would have exactly one order and `frequency` would lose its meaning. That is why segments are computed per `customer_unique_id`, and 2,997 buyers really have `n_orders >= 2`.

`olist_customer_dim` is needed because `customer_unique_id` is not unique in `olist_customers`, so it cannot be referenced by a foreign key. The table has one row per buyer with a primary key and keeps their latest `customer_id`.

`reference_date` is taken as the day after the last order in the dataset (2018-10-18) instead of `CURRENT_DATE`. The dataset is historical: with `CURRENT_DATE` all buyers would have a recency of thousands of days and the result would change with the day of the run.

`ON CONFLICT (customer_unique_id) DO UPDATE` together with `UNIQUE` guarantees one row per buyer. A repeat run updates the segment and `assigned_at` instead of adding a copy: after two runs the table has 96,096 rows. The `pg_constraint` check shows the PK, `UNIQUE`, the FK and five `CHECK`s.

## 5. JOIN queries

### J1. INNER JOIN

**Business-question:** Which product categories and seller states bring the largest revenue in delivered orders?

In [87]:
query('''
SELECT
  COALESCE(t.product_category_name_english, p.product_category_name, '(unknown)') AS category,
  s.seller_state,
  COUNT(DISTINCT oi.order_id) AS n_orders,
  SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o ON o.order_id = oi.order_id
JOIN olist_products AS p ON p.product_id = oi.product_id
JOIN olist_sellers AS s ON s.seller_id = oi.seller_id
LEFT JOIN olist_category_translation AS t ON t.product_category_name = p.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category, s.seller_state
ORDER BY revenue DESC
LIMIT 10
''')

,category,seller_state,n_orders,revenue
0,bed_bath_table,SP,8214,1070048.14
1,watches_gifts,SP,4494,1016168.80
2,health_beauty,SP,5684,790967.66
3,sports_leisure,SP,4772,680165.54
4,furniture_decor,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,housewares,SP,4021,471810.66
7,auto,SP,3018,438833.22
8,computers_accessories,SP,2783,388506.42
9,baby,SP,1981,339129.74


**Interpretation:** All ten largest combinations belong to sellers from São Paulo. bed_bath_table (1.07 million BRL) and watches_gifts (1.02 million BRL) lead, and watches_gifts has half as many orders, which means a much higher average receipt. The `INNER JOIN` of four tables is correct here because revenue is computed at item level and reviews are not involved.

### J2. LEFT JOIN + COALESCE

**Business-question:** Which customer-level features can be built for a real buyer without multiplying the review score?

In [91]:
query('''
WITH order_totals AS (
  SELECT o.order_id, c.customer_unique_id, SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
  FROM olist_orders AS o
  JOIN olist_customers AS c ON c.customer_id = o.customer_id
  LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
  GROUP BY o.order_id, c.customer_unique_id
),
review_per_order AS (
  SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
  FROM olist_order_reviews AS rv
  WHERE rv.review_score IS NOT NULL
  GROUP BY rv.order_id
),
customer_stats AS (
  SELECT
    ot.customer_unique_id,
    COUNT(DISTINCT ot.order_id) AS n_orders,
    COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
    AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score,
    COUNT(rpo.order_id) AS reviewed_orders
  FROM order_totals AS ot
  LEFT JOIN review_per_order AS rpo ON rpo.order_id = ot.order_id
  GROUP BY ot.customer_unique_id
)
SELECT
  cd.customer_unique_id,
  cd.customer_state,
  COALESCE(cs.n_orders, 0) AS n_orders,
  COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
  cs.avg_review_score,
  COALESCE(cs.reviewed_orders, 0) AS reviewed_orders,
  (cs.avg_review_score IS NULL) AS has_no_review
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC
LIMIT 10
''')

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score,reviewed_orders,has_no_review
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00,1,False
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00,2,False
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00,1,False
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00,1,False
4,459bef486812aa25204be022145caa62,ES,1,6922.21,None,0,True
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00,1,False
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00,1,False
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00,1,False
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00,1,False
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00,1,False


**Interpretation:** The biggest buyer spent 13,664 BRL in a single order and gave a rating of 1, and four of the top 10 by spend have a rating of 1. So large receipts do not guarantee satisfaction, and `avg_review_score` carries a separate signal. `COALESCE` fills counts and totals with zeros for buyers without data, while a missing review stays `NULL` with a separate flag `has_no_review`, because 0 would be a false rating. `review_per_order` aggregates reviews to the order, so several items do not multiply the score.

### J3. FULL OUTER JOIN

**Business-question:** Which states are present only among customers, only among sellers, or in both groups?

In [92]:
query('''
WITH customer_states AS (
  SELECT customer_state AS state, COUNT(*) AS n_customers
  FROM olist_customer_dim
  GROUP BY customer_state
),
seller_states AS (
  SELECT seller_state AS state, COUNT(*) AS n_sellers
  FROM olist_sellers
  GROUP BY seller_state
)
SELECT
  COALESCE(c.state, s.state) AS state,
  c.n_customers,
  s.n_sellers,
  CASE
    WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
    WHEN c.state IS NOT NULL THEN 'customers_only'
    ELSE 'sellers_only'
  END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s ON s.state = c.state
ORDER BY state_presence DESC, c.n_customers DESC NULLS LAST
''')

,state,n_customers,n_sellers,state_presence
0,AL,400,NaN,customers_only
1,TO,273,NaN,customers_only
2,AP,67,NaN,customers_only
3,RR,45,NaN,customers_only
4,SP,40296,1849.0,both
5,RJ,12381,171.0,both
6,MG,11252,244.0,both
7,RS,5276,129.0,both
8,PR,4879,349.0,both
9,SC,3531,190.0,both


**Interpretation:** Customers exist in all 27 states, while sellers are in only 23: AL, TO, AP and RR have customers but no sellers. The seller market is highly concentrated: SP has 1,849 of the 3,095 sellers. `FULL OUTER JOIN` is needed here to see both sides of the mismatch; there are no `sellers_only` rows.

### J4. SELF JOIN

**Business-question:** After how many days do real buyers come back for their next order?

In [104]:
query('''
WITH customer_orders AS (
  SELECT c.customer_unique_id, o.order_id, o.order_purchase_timestamp
  FROM olist_orders AS o
  JOIN olist_customers AS c ON c.customer_id = o.customer_id
)
SELECT
  first_order.customer_unique_id,
  first_order.order_purchase_timestamp AS first_purchase,
  next_order.order_purchase_timestamp AS next_purchase,
  (next_order.order_purchase_timestamp::DATE - first_order.order_purchase_timestamp::DATE) AS days_between
FROM customer_orders AS first_order
JOIN customer_orders AS next_order
  ON next_order.customer_unique_id = first_order.customer_unique_id
  AND next_order.order_purchase_timestamp > first_order.order_purchase_timestamp
WHERE NOT EXISTS (
  SELECT 1
  FROM customer_orders AS middle
  WHERE middle.customer_unique_id = first_order.customer_unique_id
    AND middle.order_purchase_timestamp > first_order.order_purchase_timestamp
    AND middle.order_purchase_timestamp < next_order.order_purchase_timestamp
)
ORDER BY days_between DESC
LIMIT 10
''')

,customer_unique_id,first_purchase,next_purchase,days_between
0,ccafc1c3f270410521c3c6f3b249870f,2016-10-06 19:33:34,2018-06-07 19:03:12,609
1,d8f3c4f441a9b59a29f977df16724f38,2017-01-18 21:08:18,2018-08-24 17:52:59,583
2,94e5ea5a8c1bf546db2739673060c43f,2016-10-05 21:10:56,2018-05-09 13:49:19,581
3,87b3f231705783eb2217e25851c0a45d,2016-10-08 18:45:34,2018-05-04 11:14:37,573
4,4e23e1826902ec9f208e8cc61329b494,2016-10-05 12:32:55,2018-03-13 22:28:21,524
5,a1c61f8566347ec44ea37d22854634a1,2017-03-19 08:36:36,2018-08-25 11:01:56,524
6,a262442e3ab89611b44877c7aaf77468,2017-02-17 18:27:58,2018-07-24 16:44:34,522
7,5eefb861d4921a3e628bbc65c50a480a,2017-03-05 21:47:58,2018-08-05 15:48:29,518
8,18bc87094128bbfe943cf88adcf72059,2017-01-26 23:48:56,2018-06-25 12:06:07,515
9,7e7301841ddb4064c2f3a31e4c154932,2017-02-15 12:56:00,2018-07-14 19:38:26,514


**Interpretation:** The self-join of `customer_orders` with itself finds, for each order, the next order of the same buyer, and `NOT EXISTS` removes the intermediate ones. The longest returns take 500–600 days, meaning the buyer came back after a year and a half. For a churn model such an interval is a natural feature.

### J5. SELF JOIN (summary)

**Business-question:** How is the interval between consecutive orders of the same buyer distributed?

In [121]:
query('''
WITH customer_orders AS (
  SELECT c.customer_unique_id, o.order_id, o.order_purchase_timestamp
  FROM olist_orders AS o
  JOIN olist_customers AS c ON c.customer_id = o.customer_id
),
next_orders AS (
  SELECT
    a.customer_unique_id,
    MIN(b.order_purchase_timestamp) - a.order_purchase_timestamp AS gap
  FROM customer_orders AS a
  JOIN customer_orders AS b
    ON b.customer_unique_id = a.customer_unique_id
    AND b.order_purchase_timestamp > a.order_purchase_timestamp
  GROUP BY a.customer_unique_id, a.order_id, a.order_purchase_timestamp
)
SELECT
  COUNT(*) AS repeat_pairs,
  COUNT(DISTINCT customer_unique_id) AS repeat_customers,
  COUNT(*) FILTER (WHERE gap < INTERVAL '1 day') AS same_day_repeats,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM gap) / 86400)::NUMERIC(8, 1) AS median_gap_days
FROM next_orders
''')

,repeat_pairs,repeat_customers,same_day_repeats,median_gap_days
0,3074,2740,711,38.6


**Interpretation:** There are 3,074 consecutive order pairs among 2,740 repeat buyers, with a median interval of 38.6 days. 711 repeats (23%) happened on the same day, so part of the "repeat purchases" is actually one basket split into several orders. For the frequency feature such pairs should be merged.

### J6. Anti-join

**Business-question:** Which orders have no review at all, and in which statuses are they?

In [122]:
query('''
SELECT
  o.order_status,
  COUNT(*) AS orders_without_review,
  MIN(o.order_purchase_timestamp) AS first_purchase,
  MAX(o.order_purchase_timestamp) AS last_purchase
FROM olist_orders AS o
WHERE NOT EXISTS (
  SELECT 1
  FROM olist_order_reviews AS rv
  WHERE rv.order_id = o.order_id
)
GROUP BY o.order_status
ORDER BY orders_without_review DESC
''')

,order_status,orders_without_review,first_purchase,last_purchase
0,delivered,646,2016-10-05 20:20:59,2018-08-27 17:01:43
1,shipped,75,2016-10-05 16:57:30,2018-08-07 21:40:46
2,canceled,20,2017-02-12 13:41:41,2018-09-06 10:48:12
3,unavailable,14,2017-02-06 08:18:30,2018-08-11 21:38:00
4,processing,6,2017-05-27 17:48:11,2017-12-28 12:46:11
5,invoiced,5,2017-04-24 13:37:12,2018-05-16 14:57:49
6,created,2,2017-12-05 01:07:58,2018-02-09 17:21:04


**Interpretation:** 768 orders have no review. 646 of them have status delivered, meaning the customer received the goods but did not rate them. The rest are shipped, canceled and unavailable, where a review is logically not expected. For a satisfaction model a missing review is a separate category, not an average rating.

## 6. GROUP BY, HAVING and aggregates

### A1

**Business-question:** Which categories bring more than 100,000 BRL of revenue in delivered orders, and what is their average item price?

In [123]:
query('''
SELECT
  COALESCE(t.product_category_name_english, p.product_category_name, '(unknown)') AS category,
  COUNT(DISTINCT oi.order_id) AS n_orders,
  SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
  SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
  AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price,
  ROUND(SUM(oi.freight_value) * 100.0 / SUM(oi.price), 1) AS freight_pct
FROM olist_order_items AS oi
JOIN olist_orders AS o ON o.order_id = oi.order_id
JOIN olist_products AS p ON p.product_id = oi.product_id
LEFT JOIN olist_category_translation AS t ON t.product_category_name = p.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category
HAVING SUM(oi.price) > 100000
ORDER BY gross_revenue DESC
''')

,category,n_orders,gross_revenue,freight,avg_item_price,freight_pct
0,health_beauty,8647,1233131.72,178957.81,130.28,14.5
1,watches_gifts,5495,1166176.98,98156.14,199.04,8.4
2,bed_bath_table,9272,1023434.76,201774.50,93.44,19.7
3,sports_leisure,7530,954852.55,163404.36,113.25,17.1
4,computers_accessories,6530,888724.61,143999.16,116.26,16.2
5,furniture_decor,6307,711927.69,168402.23,87.25,23.7
6,housewares,5743,615628.69,142763.56,90.60,23.2
7,cool_stuff,3559,610204.10,81476.79,164.12,13.4
8,auto,3810,578966.65,90488.10,139.85,15.6
9,toys,3804,471286.48,75774.58,116.94,16.1


**Interpretation:** `HAVING SUM(oi.price) > 100000` keeps 27 categories out of more than 70. The leader is health_beauty (1.23 million BRL), followed by watches_gifts, which has the highest average price among the leaders (199 BRL) and the smallest delivery share (8.4%). In bed_bath_table delivery makes up 19.7% of the price, so logistics weighs noticeably more. Products without a category still bring 171 thousand BRL, so `(unknown)` cannot simply be thrown away.

### A2

**Business-question:** What are the average order value (AOV) and the late-delivery share in customer states with at least 1,000 orders?

In [127]:
query('''
WITH order_totals AS (
  SELECT o.order_id, c.customer_state, o.is_late, SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
  FROM olist_orders AS o
  JOIN olist_customers AS c ON c.customer_id = o.customer_id
  JOIN olist_order_items AS oi ON oi.order_id = o.order_id
  WHERE o.order_status = 'delivered'
  GROUP BY o.order_id, c.customer_state, o.is_late
)
SELECT
  customer_state,
  COUNT(*) AS n_orders,
  SUM(order_total)::NUMERIC(12, 2) AS gmv,
  (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov,
  ROUND(AVG(is_late::INT) * 100, 1) AS late_pct
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 1000
ORDER BY late_pct DESC
''')

,customer_state,n_orders,gmv,aov,late_pct
0,CE,1279,266436.77,208.32,13.8
1,BA,3256,591137.81,181.55,12.2
2,RJ,12350,2055401.57,166.43,12.1
3,ES,1995,317657.93,159.23,10.7
4,PE,1593,308972.05,193.96,9.6
5,SC,3546,595127.78,167.83,8.2
6,GO,1957,334212.35,170.78,6.5
7,RS,5345,861472.79,161.17,6.1
8,DF,2080,346123.35,166.41,5.7
9,MG,11354,1818891.67,160.20,4.6


**Interpretation:** Among states with 1,000+ delivered orders the highest late share is in CE (13.8%), BA (12.2%) and RJ (12.1%), and the lowest in PR (4.0%) and SP (4.5%), closer to the main seller base. Remote states also have a higher AOV (CE 208 BRL versus SP 142) because delivery is more expensive there. `HAVING COUNT(*) >= 1000` cuts off small states where the percentage is unstable.

### A3

**Business-question:** Which categories were bought by customers with three or more orders?

In [130]:
query('''
SELECT
  c.customer_unique_id,
  COUNT(DISTINCT o.order_id) AS n_orders,
  SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
  STRING_AGG(
    DISTINCT COALESCE(t.product_category_name_english, p.product_category_name, '(unknown)'),
    ', ' ORDER BY COALESCE(t.product_category_name_english, p.product_category_name, '(unknown)')
  ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
LEFT JOIN olist_category_translation AS t ON t.product_category_name = p.product_category_name
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 3
ORDER BY n_orders DESC, total_spend DESC
LIMIT 10
''')

,customer_unique_id,n_orders,total_spend,categories
0,8d50f5eadf50201ccdcedfb9e2ac8455,16,729.62,"(unknown), costruction_tools_tools, fashion_ba..."
1,3e43e6105506432c953e165fb2acf44c,9,1000.85,"bed_bath_table, computers_accessories, furnitu..."
2,1b6c7548a2a1f9037c1fd3ddfed95f33,7,809.21,"furniture_decor, health_beauty, housewares, pe..."
3,ca77025e7201e3b30c44b472ff346268,7,806.61,"bed_bath_table, cool_stuff, furniture_decor, h..."
4,6469f99c1f9dfae7733b25662e7f1782,7,664.20,bed_bath_table
5,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,836.60,"auto, bed_bath_table, cool_stuff, health_beaut..."
6,dc813062e0fc23409cd255f7f53c7074,6,687.76,"cool_stuff, toys, watches_gifts"
7,63cfc61cee11cbe306bff5857d00bfe4,6,579.81,"(unknown), bed_bath_table, construction_tools_..."
8,f0e310a6839dce9de1638e0fe5ab282a,6,438.09,"(unknown), bed_bath_table, housewares"
9,12f5d6e1cbf93dafd9dcc19095df0b3d,6,58.40,"fashion_bags_accessories, furniture_bedroom, p..."


**Interpretation:** The most active buyer has 16 orders worth 730 BRL across several categories, and among buyers with 6+ orders there are single-category ones too (only bed_bath_table). `STRING_AGG(DISTINCT ... ORDER BY ...)` collects categories into one deterministic string that can be used as a "breadth of interests" feature or for recommendations.

### WHERE vs HAVING

In query A2, `WHERE o.order_status = 'delivered'` works before aggregation: it removes individual order rows, so canceled orders do not enter either GMV or the count. `HAVING COUNT(*) >= 1000` works after `GROUP BY` and removes whole state groups that have too few orders. The condition on `COUNT(*)` cannot be moved to `WHERE`, because at the `WHERE` stage groups and aggregates do not exist yet.

## 7. Set operations and window function

### S1. EXCEPT

**Business-question:** Which Portuguese product categories have no English translation in the lookup?

In [134]:
query('''
SELECT p.product_category_name AS category
FROM olist_products AS p
WHERE p.product_category_name IS NOT NULL
EXCEPT
SELECT t.product_category_name
FROM olist_category_translation AS t
ORDER BY category
''')

,category
0,pc_gamer
1,portateis_cozinha_e_preparadores_de_alimentos


**Interpretation:** Two Portuguese categories (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) have no translation, so in the queries above they are shown in Portuguese through `COALESCE`. `EXCEPT` is the natural choice here: what is needed is the set difference "present in products but missing from the lookup", and it removes duplicates by itself.

### S2. INTERSECT

**Business-question:** Which sellers had late deliveries in 2018 and are also in the bronze tier?

In [135]:
query('''
WITH late_2018 AS (
  SELECT oi.seller_id
  FROM olist_order_items AS oi
  JOIN olist_orders AS o ON o.order_id = oi.order_id
  WHERE o.is_late AND o.order_purchase_timestamp >= TIMESTAMP '2018-01-01'
),
late_and_bronze AS (
  SELECT seller_id FROM late_2018
  INTERSECT
  SELECT seller_id FROM seller_score WHERE tier = 'bronze'
)
SELECT
  (SELECT COUNT(DISTINCT seller_id) FROM late_2018) AS late_sellers_2018,
  (SELECT COUNT(*) FROM seller_score WHERE tier = 'bronze') AS bronze_sellers,
  (SELECT COUNT(*) FROM late_and_bronze) AS late_and_bronze
''')

,late_sellers_2018,bronze_sellers,late_and_bronze
0,970,539,115


**Interpretation:** In 2018, 970 sellers had late deliveries, and the bronze tier has 539. The intersection is 115 sellers: they both delay delivery and have a low rating, so they are the first candidates for `seller_alerts` with type `low_rating`. `INTERSECT` is more precise than `JOIN` here because it returns each seller once, however many late items they had.

### W1. RANK() OVER

**Business-question:** Which category leads by revenue in each seller state, and what share of the state does it take?

In [169]:
query('''
WITH category_revenue AS (
  SELECT
    s.seller_state,
    COALESCE(t.product_category_name_english, p.product_category_name, '(unknown)') AS category,
    SUM(oi.price)::NUMERIC(12, 2) AS revenue
  FROM olist_order_items AS oi
  JOIN olist_sellers AS s ON s.seller_id = oi.seller_id
  JOIN olist_products AS p ON p.product_id = oi.product_id
  LEFT JOIN olist_category_translation AS t ON t.product_category_name = p.product_category_name
  GROUP BY s.seller_state, category
),
ranked AS (
  SELECT
    seller_state,
    category,
    revenue,
    RANK() OVER (PARTITION BY seller_state ORDER BY revenue DESC) AS category_rank,
    ROUND(revenue * 100.0 / SUM(revenue) OVER (PARTITION BY seller_state), 1) AS pct_of_state
  FROM category_revenue
)
SELECT seller_state, category, revenue, pct_of_state
FROM ranked
WHERE category_rank = 1
ORDER BY revenue DESC
LIMIT 10
''')

,seller_state,category,revenue,pct_of_state
0,SP,watches_gifts,971086.60,11.1
1,PR,computers_accessories,201758.43,16.0
2,RJ,health_beauty,183336.32,21.7
3,MG,computers_accessories,172808.63,17.1
4,BA,computers,171461.75,60.0
5,SC,health_beauty,79382.13,12.6
6,RS,cool_stuff,61405.52,16.2
7,PE,computers_accessories,55977.73,61.2
8,MA,health_beauty,36408.95,100.0
9,ES,garden_tools,24799.66,52.0


**Interpretation:** `RANK() OVER (PARTITION BY seller_state ...)` computes the category rank inside a state, and `SUM(...) OVER (PARTITION BY ...)` computes the state share without a separate subquery. In SP the leader watches_gifts takes only 11% of revenue, meaning the assortment is wide. In BA and PE one category gives 60%, and in MA 100%: there are effectively one or two large sellers. For a model this is a signal that the feature "seller state" partly encodes a specific seller.

## 8. Reflection

**The hardest JOIN** was `FULL OUTER JOIN`: both sides must be aggregated to the state level beforehand, otherwise a join of millions of pairs gives false counts. The hardest business question to formulate was for the self-join: "seller pairs" give the business nothing, so I replaced them with the interval between a buyer's repeat orders. Overall the most attention went to granularity: in every query with reviews or order totals a separate CTE first reduces the data to one level, and only then are they joined.

**The most useful feature for ETL** is `ON CONFLICT`: thanks to it loads can be re-run without duplicates and without a manual `DELETE`. `RETURNING` complements it with a ready audit log.

**Window functions** in a real project I would use for deduplication (`ROW_NUMBER` for the latest review or address), for "previous order" features and intervals (`LAG`), for rolling averages of seller revenue and for ranking categories.

**Data quality:** 789 `review_id` values repeat across different orders, 547 orders have several reviews, 59% of reviews have no text, 610 products have no category, two categories have no translation, 8 orders with status delivered have no delivery date, and 23% of repeat purchases were made on the same day.

**`customer_unique_id`** is a real buyer, while `customer_id` is created per order. With `customer_id` frequency is always 1, and the 2,997 repeat buyers would disappear.

**`seller_id × order_id`** is needed because an order item multiplies a review: the naive join gave 112,372 ratings instead of 99,247 and shifted the average for 1,264 sellers.